In [10]:
from pathlib import Path
from pprint import pprint
from xml.etree import ElementTree as ET

import pymupdf4llm
import requests
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models
from qdrant_client.models import PointStruct
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("../data")
CHUNK_SIZE = 500
QDRANT_URL = "http://localhost:6333"
GROBID_URL = "http://localhost:8070"
COLLECTION_NAME = "arxiv_papers"
DENSE_MODEL = "BAAI/bge-small-en"
SPARSE_MODEL = "qdrant/bm25"
EMBEDDING_DIM = 384

pdf_files = [DATA_DIR / "1810.04805.pdf", DATA_DIR / "2302.13971.pdf"]
print(f"{len(pdf_files)} PDFs selected")
for f in pdf_files:
    print(f"  {f.name}")

2 PDFs selected
  1810.04805.pdf
  2302.13971.pdf


In [2]:
NS = "{http://www.tei-c.org/ns/1.0}"


def parse_ref(bibl):
    authors = []
    for a in bibl.findall(f".//{NS}author/{NS}persName"):
        first = a.findtext(f"{NS}forename", default="")
        last = a.findtext(f"{NS}surname", default="")
        authors.append(f"{first} {last}".strip())

    title = ""
    for parent_tag in [f"{NS}analytic", f"{NS}monogr"]:
        el = bibl.find(f"{parent_tag}/{NS}title")
        if el is not None and el.text:
            title = el.text.strip()
            break

    venue = ""
    monogr_title = bibl.find(f"{NS}monogr/{NS}title")
    if (
        monogr_title is not None
        and monogr_title.text
        and monogr_title.text.strip() != title
    ):
        venue = monogr_title.text.strip()

    year = None
    date_el = bibl.find(f".//{NS}date[@when]")
    if date_el is not None:
        try:
            year = int(date_el.get("when", "")[:4])
        except ValueError, TypeError:
            pass

    doi = None
    doi_el = bibl.find(f".//{NS}idno[@type='DOI']")
    if doi_el is not None and doi_el.text:
        doi = doi_el.text.strip()

    return {
        "authors": "; ".join(authors),
        "title": title,
        "venue": venue,
        "year": year,
        "doi": doi,
    }


def extract_references(pdf_path: Path) -> list[dict]:
    with open(pdf_path, "rb") as f:
        resp = requests.post(
            f"{GROBID_URL}/api/processFulltextDocument",
            files={"input": (pdf_path.name, f, "application/pdf")},
            data={"generateIDs": "1", "consolidateHeader": "1"},
            timeout=120,
        )
    if resp.status_code != 200:
        print(f"  GROBID failed for {pdf_path.name} (status={resp.status_code})")
        return []

    root = ET.fromstring(resp.text)
    return [parse_ref(b) for b in root.findall(f".//{NS}listBibl/{NS}biblStruct")]


paper_refs: dict[str, list[dict]] = {}
for pdf_path in pdf_files:
    stem = pdf_path.stem
    refs = extract_references(pdf_path)
    paper_refs[stem] = refs
    print(f"{stem}: {len(refs)} references")

print(
    f"\nTotal: {sum(len(r) for r in paper_refs.values())} references from {len(paper_refs)} papers"
)

1810.04805: 55 references
2302.13971: 54 references

Total: 109 references from 2 papers


In [3]:
all_chunks = []
for pdf_path in pdf_files:
    md_text = pymupdf4llm.to_markdown(str(pdf_path))
    arxiv_id = pdf_path.stem
    for i in range(0, len(md_text), CHUNK_SIZE):
        all_chunks.append({"text": md_text[i : i + CHUNK_SIZE], "arxiv_id": arxiv_id})

print(f"Parsed {len(pdf_files)} papers → {len(all_chunks)} chunks")

Parsed 2 papers → 319 chunks


In [4]:
for arxiv_id, refs in paper_refs.items():
    print(f"\n{arxiv_id}: {len(refs)} references")
    for r in refs[:3]:
        print(f"  • {r['authors'][:50]} — {r['title'][:60]} ({r['year']})")
    if len(refs) > 3:
        print(f"  ... and {len(refs) - 3} more")


1810.04805: 55 references
  • Alan Akbik; Duncan Blythe; Roland Vollgraf — Contextual string embeddings for sequence labeling (2018)
  • Rami Al-Rfou; Dokook Choe; Noah Constant; Mandy Gu — Character-level language modeling with deeper self-attention (2018)
  • Rie Kubota; Ando; Tong Zhang — A framework for learning predictive structures from multiple (2005)
  ... and 52 more

2302.13971: 54 references
  • Jacob Austin; Augustus Odena; Maxwell Nye; Maarten —  (2021)
  • Frederick Lalit R Bahl; Robert Jelinek; Mercer — A maximum likelihood approach to continuous speech recogniti (1983)
  • Yoshua Bengio; Réjean Ducharme; Pascal Vincent — A neural probabilistic language model. Advances in neural in (2000)
  ... and 51 more


In [5]:
qdrant = QdrantClient(url=QDRANT_URL)

if not qdrant.collection_exists(COLLECTION_NAME):
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=EMBEDDING_DIM, distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )
    print(f"Created collection '{COLLECTION_NAME}'")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists")

Created collection 'arxiv_papers'


In [6]:
texts = [c["text"] for c in all_chunks]

dense_encoder = SentenceTransformer(DENSE_MODEL)
dense_embeddings = dense_encoder.encode(texts, show_progress_bar=True)

bm25_encoder = SparseTextEmbedding(model_name=SPARSE_MODEL)
sparse_embeddings = list(bm25_encoder.embed(texts))

print(f"Dense shape: {dense_embeddings.shape}")
print(f"Sparse vectors: {len(sparse_embeddings)}")

Batches: 100%|██████████| 10/10 [00:06<00:00,  1.50it/s]

Dense shape: (319, 384)
Sparse vectors: 319


In [7]:
points = []
for idx, (chunk, dense_vec, sparse_vec) in enumerate(
    zip(all_chunks, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={
            "text": chunk["text"],
            "arxiv_id": chunk["arxiv_id"],
            "chunk_idx": idx,
        },
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(),
                values=sparse_vec.values.tolist(),
            ),
        },
    )
    points.append(point)

qdrant.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=64)
print(f"Uploaded {len(points)} chunks from {len(pdf_files)} papers")

Uploaded 319 chunks from 2 papers


In [11]:
query = "How many GPU hours did it take to train LLaMA-65B?"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)

pprint(results.points)

[ScoredPoint(id=213, version=4, score=1.0, payload={'text': 'GPU-hours|Total power<br>consumption|Carbon emitted<br>(tCO2eq)|\n|---|---|---|---|---|---|\n|OPT-175B|A100-80GB|400W|809,472|356 MWh|137|\n|BLOOM-175B|A100-80GB|400W|1,082,880|475 MWh|183|\n|LLaMA-7B|A100-80GB|400W|82,432|36 MWh|14|\n|LLaMA-13B|A100-80GB|400W|135,168|59 MWh|23|\n|LLaMA-33B|A100-80GB|400W|530,432|233 MWh|90|\n|LLaMA-65B|A100-80GB|400W|1,022,362|449 MWh|173|\n\n\n\nTable 15: **Carbon footprint of training different models in the same data center.** We follow Wu et al. (2022) to compute carb', 'arxiv_id': '2302.13971', 'chunk_idx': 212}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=169, version=3, score=0.45, payload={'text': 'answer the question. In Table 4, we report performance on NaturalQuestions, and in Table 5, we report on TriviaQA. On both benchmarks, LLaMA-65B achieve state-of-the-arts performance in the zero-shot and few-shot settings. More importantly, the LLaMA-13B is also competi